# 01. Setup & Cleaning
In questo notebook viene implementata la pipeline iniziale per il caricamento e la pulizia dei dati relativi al mondo degli anime. I dati 'sanificati' vengono poi salvati nella cartella `data/processed/` per le successive fasi di analisi e visualizzazione.

### Obbiettivi
- **Caricamento** dei CSV: importazione dei file necessari per l'analisi.
- **Ispezione e pulizia dei dati**: ricerca di anomalie e rimozione dei duplicati.
- **Gestione valori nulli e tipi di dato**: trattamento dei dati mancanti e correzione dei formati.
- **Salvataggio dei CSV puliti**: Esportazione dei dati pronti all'uso nella cartella `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
from functions.utils import *

## 1. Caricamento Dataset

In questa sezione carichiamo i file CSV originali per poi organizzarli all'interno di un dizionario; questo approccio ci permette di lavorare in modo più efficace durante le fasi di pulizia e ispezione, facilitando la gestione simultanea di tutti i dataset.

In [ ]:
characters_df = pd.read_csv("../data/datasets/characters.csv")
character_anime_works_df = pd.read_csv("../data/datasets/character_anime_works.csv")
character_nicknames_df = pd.read_csv("../data/datasets/character_nicknames.csv")
details_df = pd.read_csv("../data/datasets/details.csv")
favs_df = pd.read_csv("../data/datasets/favs.csv")
person_alternate_names_df = pd.read_csv("../data/datasets/person_alternate_names.csv")
person_anime_works_df = pd.read_csv("../data/datasets/person_anime_works.csv")
person_voice_works_df = pd.read_csv("../data/datasets/person_voice_works.csv")
person_details_df = pd.read_csv("../data/datasets/person_details.csv")
profiles_df = pd.read_csv("../data/datasets/profiles.csv")
ratings_df = pd.read_csv("../data/datasets/ratings.csv")
recommendations_df = pd.read_csv("../data/datasets/recommendations.csv")
stats_df = pd.read_csv("../data/datasets/stats.csv")

Stampiamo la dimensione di ciascun dataset per verificare l'esito dell'importazione.

In [ ]:
datasets = {
    "Characters": characters_df,
    "Character_anime_works": character_anime_works_df,
    "Character_nicknames": character_nicknames_df,
    "Details": details_df,
    "Favs": favs_df,
    "Person_alternate_names": person_alternate_names_df,
    "Person_anime_works": person_anime_works_df,
    "Person_voice_works": person_voice_works_df,
    "Person_details": person_details_df,
    "Profiles": profiles_df,
    "Ratings": ratings_df,
    "Recommendations": recommendations_df,
    "Stats": stats_df
}

for name, df in datasets.items():
     show_shape(df, name)

## 2. Ispezione Dataset

Visualizziamo le prime righe di `Characters`, `Details` e `Profiles` come campioni rappresentativi per capire la struttura generale dei dati.

In [ ]:
display(datasets["Characters"].head())
display(datasets["Details"].head())
display(datasets["Profiles"].head())

## 3. Rimozione dei Duplicati

In questa sezione analizziamo l'integrità dei dati cercando eventuali righe duplicate in tutti i dataset. Sfruttiamo il dizionario creato in precedenza per applicare la funzione `check_duplicates` in modo iterativo; questo ci permette di avere una panoramica immediata dei duplicati presenti e di procedere alla loro rimozione.

In [ ]:
for name, df in datasets.items():
    check_duplicates(df, name)
    print()

Successivamente, utilizziamo la funzione `clean_duplicates` per rimuoverli definitivamente, garantendo che ogni dataset contenga solo informazioni univoche e che il numero di righe rimosse corrisponda alle copie rilevate prima.

In [ ]:
datasets_to_clean = [
    "Characters", 
    "Character_nicknames", 
    "Person_alternate_names", 
    "Person_voice_works", 
    "Person_details"
]

for name in datasets_to_clean:
    if name in datasets:
        old_len = len(datasets[name])
        datasets[name] = clean_duplicates(datasets[name])
        print(f"Dataset {name:<25}: new rows {len(datasets[name])}, before was {old_len} rows, removed {old_len - len(datasets[name])}")

## 4. Definizione dei Tipi di Dato (dtypes)

In questa sezione impostiamo il formato corretto per le colonne di ogni dataset.   
Iniziamo visualizzando i tipi di dato di ogni dataset utilizzando l'attributo `.dtypes`.

In [ ]:
for name, df in datasets.items():
    print(f"\nDATASET: {name}")
    print(df.dtypes)

Trasformiamo tutte le colonne contenenti date nel formato datetime, utilizzando la funzione `convert_to_datetime` per la gestione dei valori nulli e delle date non valide.

In [ ]:
datasets["Person_details"] = convert_to_datetime(datasets["Person_details"], "birthday")

datasets["Profiles"] = convert_to_datetime(datasets["Profiles"], "joined")
datasets["Profiles"] = convert_to_datetime(datasets["Profiles"], "birthday")

datasets["Details"] = convert_to_datetime(datasets["Details"], "start_date")
datasets["Details"] = convert_to_datetime(datasets["Details"], "end_date")

Successivamente, convertiamo le restanti colonne numeriche nel tipo più adatto per l'analisi statistica. Nello specifico:
- In `Details`: Le colonne 'years', 'scored_by', 'rank' ed 'episodes' vengono convertite da float a Int64. L'uso di Int64 ci permette di trattare queste colonne come interi pur mantenendo eventuali valori nulli (NaN) senza generare errori.
- In `Profiles`: Le colonne relative allo stato di visione ('watching', 'completed', ecc.) venivano lette come object a causa della virgola usata come separatore delle migliaia. Abbiamo rimosso la virgola per permettere la corretta conversione in Int64.
- In `Characters`: La colonna 'favorites' viene uniformata al tipo Int64.

In [ ]:
datasets["Details"]["year"] = datasets["Details"]["year"].astype("Int64")
datasets["Details"]["scored_by"] = datasets["Details"]["scored_by"].astype("Int64")
datasets["Details"]["rank"] = datasets["Details"]["rank"].astype("Int64")
datasets["Details"]["episodes"] = datasets["Details"]["episodes"].astype("Int64")

datasets["Characters"]["favorites"] = datasets["Characters"]["favorites"].astype("Int64")

cols_to_fix = ['watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']

for col in cols_to_fix:
    if col in datasets["Profiles"].columns:
        datasets["Profiles"][col] = pd.to_numeric(
            datasets["Profiles"][col].astype(str).str.replace(',', ''), 
            errors='coerce'
        )
        datasets["Profiles"][col] = datasets["Profiles"][col].astype("Int64")

Per concludere questa fase, visualizziamo i nuovi tipi di dato. Questo confronto finale ci permette di confermare che tutte le conversioni siano andate a buon fine.

In [ ]:
for name, df in datasets.items():
    print(f"\nDATASET: {name}")
    print(df.dtypes)

## 5. Analisi e Gestione dei Valori Nulli (NaN)

In questa fase finale della pulizia, ci occupiamo dei valori mancanti all'interno dei nostri dataset.  
Utilizziamo la funzione `check_missing` per visualizzare la percentuale dei valori NaN e identificare in quali colonne specifiche si trovano.

In [ ]:
for name, df in datasets.items():
    check_missing(df, name)
    print()

Attraverso la funzione `drop_empty_rows`, eliminiamo le righe che contengono esclusivamente valori NaN. Queste righe risultano totalmente inutili alla nostra analisi.

In [ ]:
for name, df in datasets.items():
    datasets[name] = drop_empty_rows(df, name)

Dopo aver rimosso le righe vuote la colonna 'characters_mal_id' risulta univoca. Procediamo quindi alla sua conversione definitiva nel tipo int.

In [ ]:
datasets["Characters"]['character_mal_id'] = datasets["Characters"]['character_mal_id'].astype(int)

## 6. Salvataggio dei Dataset Puliti

In quest'ultima fase della preparazione, procediamo al salvataggio dei dataset elaborati. Dopo aver rimosso i duplicati, normalizzato i tipi di dato e gestito i valori mancanti, esportiamo i file in una cartella dedicata `data/processed/`.

In [ ]:
for name, df in datasets.items():
    path = f"../data/processed/{name}_cleaned.csv"
    df.to_csv(path, index=False)
    print(f"Salvato: {path}")